In [2]:
import json
import os
import random

# original_index_path = "/home/felix/LVSM_VAE2/LVSM-VAE/assets/evaluation_index_re10k.json"
original_index_path = "/home/teampc/LVSM-VAE/assets/evaluation_index_re10k.json"

SEED = 42
random.seed(SEED)

def make_paths(path: str):
    base, ext = os.path.splitext(path)
    return (
        f"{base}_2in{ext}",
        f"{base}_6in{ext}",
        f"{base}_8in{ext}",
    )

def sample_k_between_excluding(a: int, b: int, exclude: set[int], k: int) -> list[int]:
    lo, hi = sorted((a, b))
    pool = [x for x in range(lo + 1, hi) if x not in exclude]

    if len(pool) >= k:
        return random.sample(pool, k)

    # Fallback: if range is too small, sample with replacement (still excludes targets/ctx)
    if len(pool) > 0:
        return [random.choice(pool) for _ in range(k)]

    # If literally nothing is available, return empty (and caller can warn/fail)
    return []

with open(original_index_path, "r") as f:
    data = json.load(f)

data_2in = {}
data_6in = {}
data_8in = {}

for key, item in data.items():
    if item is None: continue
    ctx = item["context"]
    tar = item["target"]

    if len(ctx) != 2 or len(tar) != 3:
        raise ValueError(f"{key}: expected 2 context + 3 target, got {len(ctx)} + {len(tar)}")

    c0, c1 = ctx
    t0, t_mid, t2 = tar

    # ---------- 2in_1out ----------
    data_2in[key] = {
        "context": [c0, c1],
        "target": [t_mid],
    }

    # Common exclude set: never sample context endpoints or any target ids
    exclude = {c0, c1, t0, t_mid, t2}

    # ---------- 6in_1out ----------
    extra_6 = sample_k_between_excluding(c0, c1, exclude, k=4)
    ctx_6 = sorted([c0, c1] + extra_6)

    if len(ctx_6) != 6:
        print(f"Warning: {key} produced {len(ctx_6)} context views for 6in (expected 6). Range may be too small.")

    data_6in[key] = {
        "context": ctx_6,
        "target": [t_mid],
    }

    # ---------- 8in_1out ----------
    extra_8 = sample_k_between_excluding(c0, c1, exclude, k=6)
    ctx_8 = sorted([c0, c1] + extra_8)

    if len(ctx_8) != 8:
        print(f"Warning: {key} produced {len(ctx_8)} context views for 8in (expected 8). Range may be too small.")

    data_8in[key] = {
        "context": ctx_8,
        "target": [t_mid],
    }

out_2in, out_6in, out_8in = make_paths(original_index_path)

with open(out_2in, "w") as f:
    json.dump(data_2in, f, indent=4)

with open(out_6in, "w") as f:
    json.dump(data_6in, f, indent=4)

with open(out_8in, "w") as f:
    json.dump(data_8in, f, indent=4)

print("Wrote:")
print(" ", out_2in)
print(" ", out_6in)
print(" ", out_8in)


Wrote:
  /home/teampc/LVSM-VAE/assets/evaluation_index_re10k_2in.json
  /home/teampc/LVSM-VAE/assets/evaluation_index_re10k_6in.json
  /home/teampc/LVSM-VAE/assets/evaluation_index_re10k_8in.json


In [8]:
print(f"original len: {len(data)}")

path_2in = "/home/felix/LVSM_VAE2/LVSM-VAE/assets/evaluation_index_re10k_2in.json"

with open(path_2in, "r") as f:
    data_2in = json.load(f)

print(f"2-in len: {len(data_2in)}")

original len: 7194
2-in len: 6474
